## 실습 환경 준비

In [6]:
%pip -q install pandas tqdm pyarrow
# pandas: 데이터 처리 도구 (csv)
# tqdm: 작업 진행률을 보여주는 도구
# pyarrow: 큰 데이터를 빠르게 읽고 저장하는 도구

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os #운영체제(OS)와 상호작용하며 파일 경로 조작이나 디렉토리 생성을 돕는 라이브러리
import json # JSON 형식의 데이터를 읽고 쓰거나 파이썬 객체로 반환하기 위해 사용
import random # 난수 생성, 리스트 요소 무작위 섞기 등 무작위성과 관련된 기능을 가져옴
import pandas as pd # 표 형태의 데이터 프레임(dataframe)을 다루는데 특화된 데이터 분석 라이브러리
from tqdm import tqdm # 반복문의 진행 상태를 시각적인 프로그레스 바를 보여주는 기능

In [8]:
import time

for i in tqdm(range(100)):
    time.sleep(0.05) #처리가 진행되는 것을 확인하는 지연 시간

100%|██████████| 100/100 [00:05<00:00, 19.69it/s]


In [9]:
import os

#  현재 경로 출력
print(f"현재 경로: {os.getcwd()}")

# data_folder라는 이름의 폴더가 없으면 새로 생성
folder_name = "data_folder"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"'{folder_name}' 폴더를 생성했습니다.")
else:
    print('이미 폴더가 존재합니다.')

현재 경로: d:\TECHUP\0323_텍스트_데이터_증강과_대규모_전처리_파이프라인
'data_folder' 폴더를 생성했습니다.


In [10]:
## 원본 데이터 준비
### instruction tuning: 이미 어느 정도 학습된 언어 모델에게“사용자의 지시를 잘 따르도록” 추가로 훈련시키는 과정
#### 모델에게 이런 질문이 들어오면 이런 식으로 답해야 한다.를 가르치기 위해 질문-답변 형태의 데이터를 많이 사용함.
base_texts = [
    "토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.",
    "서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.",
    "데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.",
    "역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.",
    "질문-답변 데이터는 instruction tuning에서 자주 활용된다.",
    "대규모 데이터 처리는 메모리와 시간 제약을 고려해야 한다.",
    "전처리 파이프라인은 ingest, clean, tokenize, filter, save 단계로 구성될 수 있다.",
    "JSONL 형식은 한 줄에 하나의 JSON 객체를 저장하는 방식이다."
]

df_base = pd.DataFrame({"text": base_texts})
df_base

,text
0,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.
1,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.
2,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.
3,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.
4,질문-답변 데이터는 instruction tuning에서 자주 활용된다.
5,대규모 데이터 처리는 메모리와 시간 제약을 고려해야 한다.
6,"전처리 파이프라인은 ingest, clean, tokenize, filter, sa..."
7,JSONL 형식은 한 줄에 하나의 JSON 객체를 저장하는 방식이다.


### 데이터 증강

- 데이터 증강이 필요한 이유


    - 데이터 수가 부족한 경우

    - 같은 의미를 다양한 표현으로 학습시키고 싶은 경우

    - 특정 태스크 형식(QA, instruction 등)으로 바꾸고 싶은 경우

In [11]:
# 1. 패러프레이즈용 간단 치환 사전
## 패러프레이즈(paraphrase)는 의미는 그대로 유지하면서 표현만 다르게 바꾸는 것
synonym_dict = {
    "과정": ["절차", "단계"],
    "처리": ["가공", "분석"],
    "작은": ["세부적인", "더 작은"],
    "다양성": ["변화 폭", "다양한 표현"],
    "사용된다": ["활용된다", "쓰인다"],
    "도움": ["기여", "지원"],
    "구성될 수 있다": ["이루어질 수 있다", "설계될 수 있다"],
    "고려해야 한다": ["반영해야 한다", "신경 써야 한다"]
}

In [12]:
# 2. 동의어 치환 함수
# text: 원본 문장, synonym_dict: {단어: [동의어]} 형태의 사전, replace_prob:치환될 확률(기본값=0.3%)
def synonym_replacement(text, synonym_dict, replace_prob=0.3):
    
    # 변경 사항을 담을 변수에 원본 텍스트를 복사하여 초기화
    new_text = text
    # synonym_dict 사전에 등록된 각 단어(word)와 그에 해당하는 동의어(synonym)를 하나씩 꺼냄.
    for word, synonyms in synonym_dict.items():
        # 1. 원본 문장에 해당 단어가 포함되어 있는지 확인하고,
        # 2. 0~1 사이의 난수를 생성하여 설정된 확률(replace_prob)보다 작은지 체크
        if word in new_text and random.random() < replace_prob:
            # 조건을 만족하면, 동의어 리스트 중 하나를 무작위 선택(random.choice)하여
            # 문장 내의 단어를 딱 한번만(count=1) 치환
            new_text = new_text.replace(word, random.choice(synonyms), 1)
            
    return new_text
    # 최종적으로 동의어가 반영된(혹은 반영되지 않은) 문장을 반환

- lambda: 이름이 없는 익명 함수, 복잡한 def 선언 없이 한줄로 간결하게 함수를 만들 때 사용

In [13]:
# (이름, 나이) 형태의 리스트
users = [('김철수',30),('이영희',25),('박민수',35) ]

#(나이 요소의 1번째 인덱스)를 기준으로 정렬
users.sort(key=lambda x:x[1])
# 파이썬에서 쓰는 것이지만, 권장 문법은 아닌데, 데이터 분석시 속도 때문에 사용.
# lambda를 함수로 짠다면? 어떻게 될까?
print(users)

[('이영희', 25), ('김철수', 30), ('박민수', 35)]


In [14]:
def get_age(user_tuple):
    return user_tuple[1]

users = [('김철수',30),('이영희',25),('박민수',35) ]

users.sort(key=get_age)
print(users)


[('이영희', 25), ('김철수', 30), ('박민수', 35)]


- map: 모든 요소에 동일한 규칙을 적용할때 유용

In [15]:
numbers = [1,2,3,4,5]
results = list(map(lambda x: x+10, numbers)) # 단, map의 결과를 보려면 list로 감싸야 함.
print(results)

[11, 12, 13, 14, 15]


In [16]:
# filter
numbers = [1,2,3,4,5]
results = list(filter(lambda x: x%2==0, numbers)) # 단, map의 결과를 보려면 list로 감싸야 함.
print(results)

[2, 4]


In [17]:
# 3. 문장 재구성 함수
## 완전한 자연스러운 paraphrase는 아니지만, 실습에서는 “표현을 바꾼다”는 개념을 보여주기에 충분

def simple_rephrase(text):
    patterns = [
        lambda x: f"다음 내용을 설명하면, {x}",
        lambda x: f"{x} 이 개념은 자연어처리에서 중요하다.",
        lambda x: f"쉽게 말해, {x}",
        lambda x: f"정리하면, {x}"
    ]
    return random.choice(patterns)(text)

In [18]:
# 4. 패러프레이즈 데이터 생성
augmented_paraphrases = []

for text in base_texts:
    aug1 = synonym_replacement(text, synonym_dict)
    aug2 = simple_rephrase(text)

    # 동의어 치환으로 만들어진 데이터를 딕셔너리 형태로 리스트에 추가함.
    augmented_paraphrases.append({
        "original": text,               # 원본 문장
        "augmented": aug1,              # 변형된 문장
        "type": "synonym_replacement"   # 사용된 기법 종류
    })

    augmented_paraphrases.append({
        "original": text,
        "augmented": aug2,
        "type": "simple_rephrase"
    })

df_para = pd.DataFrame(augmented_paraphrases)
df_para.head(10)

,original,augmented,type
0,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,synonym_replacement
1,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,"쉽게 말해, 토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.",simple_rephrase
2,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,synonym_replacement
3,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다. 이 개념은 자연어처...,simple_rephrase
4,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.,데이터 증강은 학습 데이터의 변화 폭을 높이는 데 사용된다.,synonym_replacement
5,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.,"쉽게 말해, 데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.",simple_rephrase
6,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.,synonym_replacement
7,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다. 이 개...,simple_rephrase
8,질문-답변 데이터는 instruction tuning에서 자주 활용된다.,질문-답변 데이터는 instruction tuning에서 자주 활용된다.,synonym_replacement
9,질문-답변 데이터는 instruction tuning에서 자주 활용된다.,"쉽게 말해, 질문-답변 데이터는 instruction tuning에서 자주 활용된다.",simple_rephrase


In [19]:
# 5. 설명문을 QA 데이터로 변환

def text_to_qa(text):
    return {
        "instruction": "다음 개념에 대해 질문과 답변 형식으로 작성하시오.",
        "question": f"{text.split('는')[0]}란 무엇인가?",
        "answer": text
    }

qa_data = [text_to_qa(text) for text in base_texts]
df_qa = pd.DataFrame(qa_data)
df_qa.head()

,instruction,question,answer
0,다음 개념에 대해 질문과 답변 형식으로 작성하시오.,토큰화란 무엇인가?,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.
1,다음 개념에 대해 질문과 답변 형식으로 작성하시오.,서브워드 토큰화란 무엇인가?,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.
2,다음 개념에 대해 질문과 답변 형식으로 작성하시오.,데이터 증강은 학습 데이터의 다양성을 높이란 무엇인가?,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.
3,다음 개념에 대해 질문과 답변 형식으로 작성하시오.,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리란 무엇인가?,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.
4,다음 개념에 대해 질문과 답변 형식으로 작성하시오.,질문-답변 데이터란 무엇인가?,질문-답변 데이터는 instruction tuning에서 자주 활용된다.


In [20]:
# 6. 설명문을 instruction 데이터로 변환
def text_to_instruction(text):
    return {
        "instruction": "다음 개념을 초보자도 이해할 수 있도록 설명하시오.",
        "input": text.split("는")[0],
        "output": text
    }

instruction_data = [text_to_instruction(text) for text in base_texts]
df_inst = pd.DataFrame(instruction_data)
df_inst.head()

,instruction,input,output
0,다음 개념을 초보자도 이해할 수 있도록 설명하시오.,토큰화,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.
1,다음 개념을 초보자도 이해할 수 있도록 설명하시오.,서브워드 토큰화,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.
2,다음 개념을 초보자도 이해할 수 있도록 설명하시오.,데이터 증강은 학습 데이터의 다양성을 높이,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.
3,다음 개념을 초보자도 이해할 수 있도록 설명하시오.,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.
4,다음 개념을 초보자도 이해할 수 있도록 설명하시오.,질문-답변 데이터,질문-답변 데이터는 instruction tuning에서 자주 활용된다.


### 증강 전/후 품질 점검

In [21]:
a = 100
if a >0 and (a%2)==0:
    print('a is even')

a is even


In [22]:
5 and 7 #7
5 or 7  #5

# and는 큰 수, or은 작은수가 출력됨, 그 이유는 and를 쓰면 뒤에 있는 조건이 나중에 실행되서

5

In [23]:
5 & 7 # 5
5 | 7 # 7

# '&' 과 '|'은 집합 연산자, 비트 연산자이기 때문이다.

5 & 7 # 비트(0과 1) 연산자
#5 = 4+1 = 0101 -> 2^3*0+2^2*1+2^1*0+2^0*1
#7 = 4+2+1 = 0111 -> 2^3*0+2^2*1+2^1*1+2^0*1
# 0101 -> 5

# 어디에 쓰이나요? pandas에서 많이 사용됨

5

In [24]:
# 1. 중복 여부 확인
def check_duplicates(augmented_list):
    seen = set()
    duplicates = []

    for item in augmented_list:
        text = item["augmented"]
        if text in seen:
            duplicates.append(text)
        seen.add(text)

    return duplicates

duplicates = check_duplicates(augmented_paraphrases)
print("중복 개수:", len(duplicates))
print("중복 샘플:", duplicates[:5])

중복 개수: 0
중복 샘플: []


In [25]:
# 2. 길이 변화 확인
df_para["original_len"] = df_para["original"].apply(len)
df_para["augmented_len"] = df_para["augmented"].apply(len)
df_para["length_diff"] = df_para["augmented_len"] - df_para["original_len"]

df_para[["type", "original_len", "augmented_len", "length_diff"]].head()

,type,original_len,augmented_len,length_diff
0,synonym_replacement,38,38,0
1,simple_rephrase,38,45,7
2,synonym_replacement,35,35,0
3,simple_rephrase,35,55,20
4,synonym_replacement,32,33,1


In [26]:
# 3. 사람이 직접 품질 확인할 샘플 뽑기
sample_check = df_para.sample(5, random_state=42)
sample_check

,original,augmented,type,original_len,augmented_len,length_diff
0,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,synonym_replacement,38,38,0
1,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,"쉽게 말해, 토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.",simple_rephrase,38,45,7
5,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.,"쉽게 말해, 데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.",simple_rephrase,32,39,7
14,JSONL 형식은 한 줄에 하나의 JSON 객체를 저장하는 방식이다.,JSONL 형식은 한 줄에 하나의 JSON 객체를 저장하는 방식이다.,synonym_replacement,38,38,0
13,"전처리 파이프라인은 ingest, clean, tokenize, filter, sa...","전처리 파이프라인은 ingest, clean, tokenize, filter, sa...",simple_rephrase,62,82,20


**증강 데이터셋 샘플 만들기**

In [27]:
augmented_dataset = []

for row in augmented_paraphrases:
    augmented_dataset.append({
        "task_type": row["type"],
        "source_text": row["original"],
        "generated_text": row["augmented"]
    })

for row in qa_data:
    augmented_dataset.append({
        "task_type": "qa",
        "source_text": row["question"],
        "generated_text": row["answer"]
    })

for row in instruction_data:
    augmented_dataset.append({
        "task_type": "instruction",
        "source_text": row["input"],
        "generated_text": row["output"]
    })

df_augmented = pd.DataFrame(augmented_dataset)
df_augmented.head(15)

,task_type,source_text,generated_text
0,synonym_replacement,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.
1,simple_rephrase,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,"쉽게 말해, 토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다."
2,synonym_replacement,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.
3,simple_rephrase,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다. 이 개념은 자연어처...
4,synonym_replacement,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.,데이터 증강은 학습 데이터의 변화 폭을 높이는 데 사용된다.
5,simple_rephrase,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.,"쉽게 말해, 데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다."
6,synonym_replacement,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.
7,simple_rephrase,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다.,역번역은 문장을 다른 언어로 번역한 뒤 다시 원래 언어로 되돌리는 방식이다. 이 개...
8,synonym_replacement,질문-답변 데이터는 instruction tuning에서 자주 활용된다.,질문-답변 데이터는 instruction tuning에서 자주 활용된다.
9,simple_rephrase,질문-답변 데이터는 instruction tuning에서 자주 활용된다.,"쉽게 말해, 질문-답변 데이터는 instruction tuning에서 자주 활용된다."


**증강 규칙 문서 만들기**

In [28]:
augmentation_rules = [
    {"rule_name": "synonym_replacement", "description": "일부 단어를 유사 표현으로 치환"},
    {"rule_name": "simple_rephrase", "description": "문장 앞에 설명형 표현을 붙여 재구성"},
    {"rule_name": "qa_conversion", "description": "설명문을 질문-답변 형태로 변환"},
    {"rule_name": "instruction_conversion", "description": "설명문을 instruction-output 형태로 변환"}
]

df_rules = pd.DataFrame(augmentation_rules)
df_rules

,rule_name,description
0,synonym_replacement,일부 단어를 유사 표현으로 치환
1,simple_rephrase,문장 앞에 설명형 표현을 붙여 재구성
2,qa_conversion,설명문을 질문-답변 형태로 변환
3,instruction_conversion,설명문을 instruction-output 형태로 변환


# 마크다운(Markdown) 설명과 데이터 저장 방법

## 1. 마크다운이란?

마크다운(Markdown)은 **간단한 기호를 사용해서 문서를 구조화하고 꾸밀 수 있는 텍스트 기반 문법**입니다.  
워드 같은 문서 편집기 없이도 제목, 목록, 표, 코드, 링크 등을 쉽게 작성할 수 있습니다.

마크다운은 다음과 같은 곳에서 많이 사용됩니다.

- GitHub README
- Notion
- Obsidian
- VS Code
- 블로그 초안
- 수업 정리
- 연구 노트
- 실험 기록

# 3. 기본 문법
## 3-1 제목

제목은 '#' 기호를 사용합니다. '#'의 개수가 많아질수록 제목 크기는 작아진다.

### 글자 강조

**굵게**

*기울임*

~~취소선~~

## 대규모 데이터 처리

#### 1. 대량 데이터 흉내내기

In [29]:
large_dataset = pd.concat([df_augmented] * 500, ignore_index=True)
print("전체 행 수:", len(large_dataset))
large_dataset.head()

전체 행 수: 16000


,task_type,source_text,generated_text
0,synonym_replacement,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.
1,simple_rephrase,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,"쉽게 말해, 토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다."
2,synonym_replacement,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.
3,simple_rephrase,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다. 이 개념은 자연어처...
4,synonym_replacement,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.,데이터 증강은 학습 데이터의 변화 폭을 높이는 데 사용된다.


2. csv로 저장 후 chunk 단위로 읽기

In [30]:
large_dataset.to_csv("large_augmented_dataset.csv", index=False, encoding="utf-8-sig")

In [32]:
#파이썬의 next() 함수는 iterator에서 다음 요소를 꺼내올때 사용. 반복문(for)의 내부 동작 원리에 쓰임.
fruits = ['사과','바나나','딸기']
it = iter(fruits) #itertor 객체 생성

print(next(it))
print(next(it))
print(next(it))

사과
바나나
딸기


In [33]:
chunk_iter = pd.read_csv("large_augmented_dataset.csv", chunksize=1000)

first_chunk = next(chunk_iter)
print(first_chunk.shape)
first_chunk.head()

(1000, 3)


,task_type,source_text,generated_text
0,synonym_replacement,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.
1,simple_rephrase,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,"쉽게 말해, 토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다."
2,synonym_replacement,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.
3,simple_rephrase,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다. 이 개념은 자연어처...
4,synonym_replacement,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.,데이터 증강은 학습 데이터의 변화 폭을 높이는 데 사용된다.


#### map/filter 기반 전처리 파이프라인

In [34]:
# 1. clean 함수
def clean_text(text):
    text = str(text).strip()
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text

In [35]:
# 2. 간단 토큰 수 계산 함수
## 정교한 토크나이저 대신 공백 기준으로 간단히 처리
def count_tokens_simple(text):
    return len(str(text).split())

In [36]:
# 3. map 함수 적용
def preprocess_chunk(df_chunk):
    df_chunk = df_chunk.copy()

    df_chunk["source_text"] = df_chunk["source_text"].apply(clean_text)
    df_chunk["generated_text"] = df_chunk["generated_text"].apply(clean_text)

    df_chunk["source_tokens"] = df_chunk["source_text"].apply(count_tokens_simple)
    df_chunk["generated_tokens"] = df_chunk["generated_text"].apply(count_tokens_simple)

    return df_chunk

In [37]:
# 4. filter 함수 적용
## 너무 짧거나 비어 있는 데이터는 제거
def filter_chunk(df_chunk, min_tokens=2):
    cond = (
        (df_chunk["source_text"].str.len() > 0) &
        (df_chunk["generated_text"].str.len() > 0) &
        (df_chunk["generated_tokens"] >= min_tokens)
    )
    return df_chunk[cond].reset_index(drop=True)

- 전체 파이프라인

In [38]:
processed_chunks = []

for chunk in pd.read_csv("large_augmented_dataset.csv", chunksize=1000):
    chunk = preprocess_chunk(chunk)
    chunk = filter_chunk(chunk, min_tokens=2)
    processed_chunks.append(chunk)

df_processed = pd.concat(processed_chunks, ignore_index=True)
print("전처리 후 행 수:", len(df_processed))
df_processed.head()

전처리 후 행 수: 16000


,task_type,source_text,generated_text,source_tokens,generated_tokens
0,synonym_replacement,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,10,10
1,simple_rephrase,토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.,"쉽게 말해, 토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.",10,12
2,synonym_replacement,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,10,10
3,simple_rephrase,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.,서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다. 이 개념은 자연어처...,10,14
4,synonym_replacement,데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.,데이터 증강은 학습 데이터의 변화 폭을 높이는 데 사용된다.,8,9


### JSONL(JSON Lines) 저장

- 파일 안에서 한 줄에 JSON 객체 하나씩 저장하는 형식

- LLM 학습용 데이터에서는 JSONL이 자주 쓰임

```
JSON:
[
  {"text": "안녕하세요"},
  {"text": "오늘 날씨가 좋습니다"}
]
```

```
JSONL:
{"text": "안녕하세요"}
{"text": "오늘 날씨가 좋습니다"}
```
반면 JSONL은 배열 [] 없이, 레코드 하나를 한 줄씩 저장






#### 왜 LLM 학습용 데이터에서 JSONL을 자주 쓰나?

**1. 한 줄씩 읽기 쉬움**

- 대용량 파일을 전부 메모리에 올리지 않고도
한 줄씩 순서대로 읽어서 처리할 수 있습니다.

-> 스트리밍 처리에 유리

**2. 데이터 하나가 깨져도 전체가 덜 망가짐**

- JSON 배열 파일은 중간에 형식이 조금만 깨져도
전체 파일 파싱이 실패할 수 있습니다.

-> JSONL은 각 줄이 독립적이므로, 문제 있는 줄만 건너뛰고 나머지를 계속 처리하기 쉬움.

**3. 필드 구조를 명확하게 넣을 수 있음.**

- LLM 학습 데이터는 단순 문장만 있는 것이 아니라
다양한 필드를 갖는 경우가 많습니다.



```예제
{"prompt": "자기소개를 해줘", "response": "안녕하세요. 저는 ..."}
{"instruction": "영어로 번역해줘", "input": "반갑습니다", "output": "Nice to meet you."}
{"messages": [{"role": "user", "content": "안녕"}, {"role": "assistant", "content": "안녕하세요"}]}
```



In [39]:
# 1. JSONL 저장함수
def save_to_jsonl(df, path):
    """
    Pandas 데이터프레임을 jsonl 형식으로 저장하는 함수
    - df: 저장할 데이터 프레임
    - path: 저장할 파일 경로
    
    """
    with open(path, "w", encoding="utf-8") as f:
        # 파일을 쓰기모드, 인코딩 utf-8
        for _, row in df.iterrows():
            # _: 인덱스 번호는 사용하지 않으므로, 언더 스코어로 처리
            record = row.to_dict()
            # 행 데이터를 파이썬 딕셔너리(dict) 형태로 변환.
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            # 딕셔너리를 json 문자열로 반환 (ensure_ascii=False로 한글 깨짐 방지)
            # 문자열 끝에 줄바꿈을 추가하여 파일에 기록

In [40]:
#2. 저장 실행
save_to_jsonl(df_processed, "processed_llm_dataset.jsonl")
print("저장 완료")

저장 완료


In [41]:
#3. 저장 결과 일부 확인
with open("processed_llm_dataset.jsonl", "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline().strip())

{"task_type": "synonym_replacement", "source_text": "토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.", "generated_text": "토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.", "source_tokens": 10, "generated_tokens": 10}
{"task_type": "simple_rephrase", "source_text": "토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.", "generated_text": "쉽게 말해, 토큰화는 문장을 모델이 처리할 수 있는 작은 단위로 나누는 과정이다.", "source_tokens": 10, "generated_tokens": 12}
{"task_type": "synonym_replacement", "source_text": "서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.", "generated_text": "서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.", "source_tokens": 10, "generated_tokens": 10}
{"task_type": "simple_rephrase", "source_text": "서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다.", "generated_text": "서브워드 토큰화는 희귀 단어를 더 잘 처리할 수 있도록 돕는다. 이 개념은 자연어처리에서 중요하다.", "source_tokens": 10, "generated_tokens": 14}
{"task_type": "synonym_replacement", "source_text": "데이터 증강은 학습 데이터의 다양성을 높이는 데 사용된다.", "generated_text": "데이터 증강은 학습 데이터의 변화 폭을 높이는 데 사용된다.", "source_tokens": 8, "generated_tokens": 9}


## Parquet 형식이란?

**Parquet**는 데이터를 저장하는 **파일 형식** 중 하나입니다.  
주로 **표 형태 데이터**를 효율적으로 저장하고 읽기 위해 사용합니다.

### 특징
- **열(column) 단위 저장 방식**을 사용함
- 데이터 용량을 **작게 압축**하기 좋음
- 필요한 열만 골라서 읽을 수 있어 **속도가 빠름**
- 대용량 데이터 처리에 많이 사용됨

### 왜 열 단위 저장 방식이 좋을까?
- 행을 열이 있어야지만 만들어지니까
- 가로로 가는 것이 결국 세로 하나하나의 집합이기 때문에 열 단위가 훨씬 빠르다.

### 왜 쓰나?
예를 들어 데이터에  
`이름`, `나이`, `주소`, `점수` 같은 여러 열이 있을 때,  
점수 열만 필요하면 **점수 열만 빠르게 읽을 수 있음**  
→ 그래서 분석 작업에 효율적입니다.

### 주로 쓰이는 곳
- 데이터 분석
- 머신러닝
- 대용량 데이터 처리
- Spark, Pandas, PyArrow 같은 도구


In [ ]:
# 3.parquet 형식으로도 저장
df_processed.to_parquet("processed_llm_dataset.parquet", index=False)
print("parquet 저장 완료")

In [ ]:
# 파이프라인
def run_pipeline(input_csv, output_jsonl, chunksize=1000, min_tokens=2):
    processed_chunks = []

    for chunk in pd.read_csv(input_csv, chunksize=chunksize):
        chunk = preprocess_chunk(chunk)
        chunk = filter_chunk(chunk, min_tokens=min_tokens)
        processed_chunks.append(chunk)

    final_df = pd.concat(processed_chunks, ignore_index=True)
    save_to_jsonl(final_df, output_jsonl)

    return final_df

In [ ]:
final_df = run_pipeline(
    input_csv="large_augmented_dataset.csv",
    output_jsonl="final_train_dataset.jsonl",
    chunksize=1000,
    min_tokens=2
)

final_df.head()